<a href="https://colab.research.google.com/github/kyeeun7706-coder/e-waste-location-optimization/blob/main/%EC%A0%84%EC%A2%85%EC%84%A4_0330.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

라이브러리 설치

In [3]:
!pip install geopandas osmnx networkx rasterio rasterstats shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.1 MB/s eta 0:00:00


라이브러리 임포트 + 드라이브 마운트

In [27]:
import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import networkx as nx
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Point

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


파일 경로 설정

In [6]:
BASE = '/content/drive/MyDrive/전종설'

# 지표 1 - 행정동 경계 (폴더 안 .shp)
DONG_SHP = f'{BASE}/지표1_행정구역경계/bnd_dong_11180_2025_2Q.shp'

# 지표 1 - 연령별 인구 CSV
POP_CSV = f'{BASE}/지표1_202412_202512_금천구_연령별인구현황_연간.csv'

# 지표 2 - DEM 래스터
DEM_SEOUL_IMG = f'{BASE}/지표2_서울dem.img'
DEM_ANYANG_IMG = f'{BASE}/지표2_안양dem.img'

# 지표 2 - 격자별 고령인구 (폴더 안 .shp)
GRID_OLD_SHP = f'{BASE}/지표2_금천구격자별고령인구수/vl_blk.shp'

# 지표 2 - 격자별 총인구 (폴더 안 .shp)
GRID_TOT_SHP = f'{BASE}/지표2_금천구격자별총인구수/vl_blk.shp'

# 지표 3 - 수거함 8개 위치
BINS_CSV = f'{BASE}/지표3_금천구폐가전수거함8개위치.csv'



---



파일 정상 로드 확인

In [7]:
# 행정동 경계
dong = gpd.read_file(DONG_SHP)
print("행정동 경계 컬럼:", dong.columns.tolist())
print(dong.head(3))

행정동 경계 컬럼: ['BASE_DATE', 'ADM_CD', 'ADM_NM', 'geometry']
  BASE_DATE    ADM_CD ADM_NM  \
0  20250630  11180510    가산동   
1  20250630  11180520   독산1동   
2  20250630  11180530   독산2동   

                                            geometry  
0  POLYGON ((945134.075 1943182.19, 945138.213 19...  
1  POLYGON ((946851.545 1942312.478, 946849.923 1...  
2  POLYGON ((947770.689 1941031.612, 947770.422 1...  


In [8]:
# 연령별 인구 CSV
pop = pd.read_csv(POP_CSV, encoding='EUC-KR')
print("인구 CSV 컬럼:", pop.columns.tolist())
print(pop.head(3))

인구 CSV 컬럼: ['행정구역', '2024년_거주자_총인구수', '2024년_거주자_연령구간인구수', '2024년_거주자_65~69세', '2024년_거주자_70~74세', '2024년_거주자_75~79세', '2024년_거주자_80~84세', '2024년_거주자_85~89세', '2024년_거주자_90~94세', '2024년_거주자_95~99세', '2024년_거주자_100세 이상', '2024년_남_거주자_총인구수', '2024년_남_거주자_연령구간인구수', '2024년_남_거주자_65~69세', '2024년_남_거주자_70~74세', '2024년_남_거주자_75~79세', '2024년_남_거주자_80~84세', '2024년_남_거주자_85~89세', '2024년_남_거주자_90~94세', '2024년_남_거주자_95~99세', '2024년_남_거주자_100세 이상', '2024년_여_거주자_총인구수', '2024년_여_거주자_연령구간인구수', '2024년_여_거주자_65~69세', '2024년_여_거주자_70~74세', '2024년_여_거주자_75~79세', '2024년_여_거주자_80~84세', '2024년_여_거주자_85~89세', '2024년_여_거주자_90~94세', '2024년_여_거주자_95~99세', '2024년_여_거주자_100세 이상', '2025년_거주자_총인구수', '2025년_거주자_연령구간인구수', '2025년_거주자_65~69세', '2025년_거주자_70~74세', '2025년_거주자_75~79세', '2025년_거주자_80~84세', '2025년_거주자_85~89세', '2025년_거주자_90~94세', '2025년_거주자_95~99세', '2025년_거주자_100세 이상', '2025년_남_거주자_총인구수', '2025년_남_거주자_연령구간인구수', '2025년_남_거주자_65~69세', '2025년_남_거주자_70~74세', '2025년_남_거주자_75~79세', '2025년_남_거주자_80~84세', '2025년_남_

In [9]:
# 격자별 고령인구
grid_old = gpd.read_file(GRID_OLD_SHP)
print("격자 고령인구 컬럼:", grid_old.columns.tolist())
print(grid_old.head(3))

격자 고령인구 컬럼: ['gid', 'lbl', 'val', 'geometry']
            gid   lbl  val                                           geometry
0  ë¤ì¬444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  ë¤ì¬444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  ë¤ì¬444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...


In [10]:
# 격자별 총인구
grid_tot = gpd.read_file(GRID_TOT_SHP)
print("격자 총인구 컬럼:", grid_tot.columns.tolist())
print(grid_tot.head(3))

격자 총인구 컬럼: ['gid', 'lbl', 'val', 'geometry']
            gid   lbl  val                                           geometry
0  ë¤ì¬444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  ë¤ì¬444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  ë¤ì¬444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...


In [11]:
# DEM
with rasterio.open(DEM_SEOUL_IMG) as src:
    print("DEM CRS:", src.crs)
    print("DEM 해상도:", src.res)
    print("DEM 크기:", src.width, "x", src.height)

DEM CRS: EPSG:5179
DEM 해상도: (90.0, 90.0)
DEM 크기: 253 x 316


In [12]:
# 수거함 위치
bins_df = pd.read_csv(BINS_CSV)
print("수거함 컬럼:", bins_df.columns.tolist())
print(bins_df)

수거함 컬럼: ['id', 'name', 'lat', 'lon']
   id   name        lat         lon
0   1  수거함_1  37.469710  126.898966
1   2  수거함_2  37.465816  126.897190
2   3  수거함_3  37.462436  126.898027
3   4  수거함_4  37.463940  126.891584
4   5  수거함_5  37.462466  126.890374
5   6  수거함_6  37.457927  126.897581
6   7  수거함_7  37.460229  126.886538
7   8  수거함_8  37.451069  126.897925




---



### 🚨 지표2_총인구/고령인구 한글 인코딩 깨짐 이슈 발생

격자 인코딩 문제 해결 + 데이터 확인

In [13]:
# dbf 파일을 직접 읽어서 인코딩 문제 해결
grid_old = gpd.read_file(GRID_OLD_SHP, encoding='cp949')
grid_tot = gpd.read_file(GRID_TOT_SHP, encoding='cp949')

print("격자 고령인구 샘플:")
print(grid_old.head(5))
print("\nval 컬럼 null 개수:", grid_old['val'].isna().sum())
print("val 컬럼 유효값 개수:", grid_old['val'].notna().sum())
print("\n격자 총인구 샘플:")
print(grid_tot.head(5))

격자 고령인구 샘플:
        gid   lbl  val                                           geometry
0  떎궗444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  떎궗444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  떎궗444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...
3  떎궗445427  None  NaN  POLYGON ((944500 1942700, 944500 1942800, 9446...
4  떎궗445428  None  NaN  POLYGON ((944500 1942800, 944500 1942900, 9446...

val 컬럼 null 개수: 715
val 컬럼 유효값 개수: 709

격자 총인구 샘플:
        gid   lbl  val                                           geometry
0  떎궗444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  떎궗444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  떎궗444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...
3  떎궗445427  None  NaN  POLYGON ((944500 1942700, 944500 1942800, 9446...
4  떎궗445428  None  NaN  POLYGON ((944500 1942800, 944500 1942900, 9446...


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp949 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from cp949 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


val 값 실제 확인

In [14]:
# NaN 아닌 행만 보기
print("고령인구 격자 유효값 샘플:")
print(grid_old[grid_old['val'].notna()].head(10))

print("\n총인구 격자 유효값 샘플:")
print(grid_tot[grid_tot['val'].notna()].head(10))

print("\n고령인구 val 통계:")
print(grid_old['val'].describe())

print("\n총인구 val 통계:")
print(grid_tot['val'].describe())

고령인구 격자 유효값 샘플:
         gid    lbl   val                                           geometry
14  떎궗446430    N/A   0.0  POLYGON ((944600 1943000, 944600 1943100, 9447...
21  떎궗447427    N/A   0.0  POLYGON ((944700 1942700, 944700 1942800, 9448...
23  떎궗447429    N/A   0.0  POLYGON ((944700 1942900, 944700 1943000, 9448...
33  떎궗448427   6.00   6.0  POLYGON ((944800 1942700, 944800 1942800, 9449...
42  떎궗449422  70.00  70.0  POLYGON ((944900 1942200, 944900 1942300, 9450...
43  떎궗449423  11.00  11.0  POLYGON ((944900 1942300, 944900 1942400, 9450...
56  떎궗450420    N/A   0.0  POLYGON ((945000 1942000, 945000 1942100, 9451...
58  떎궗450422    N/A   0.0  POLYGON ((945000 1942200, 945000 1942300, 9451...
61  떎궗450425    N/A   0.0  POLYGON ((945000 1942500, 945000 1942600, 9451...
65  떎궗450429    N/A   0.0  POLYGON ((945000 1942900, 945000 1943000, 9451...

총인구 격자 유효값 샘플:
         gid     lbl    val                                           geometry
14  떎궗446430     N/A    0.0  POLYGON ((944

좌표계 확인(모두 같아야 함)

In [15]:
print("행정동 경계 CRS:", dong.crs)
print("격자 고령인구 CRS:", grid_old.crs)
print("격자 총인구 CRS:", grid_tot.crs)
print("DEM CRS: EPSG:5179")

행정동 경계 CRS: EPSG:5179
격자 고령인구 CRS: EPSG:5179
격자 총인구 CRS: EPSG:5179
DEM CRS: EPSG:5179




---



# 👌**데이터 준비 끝, 지표 1,2,3 전처리 시작**

## <지표 1>
인구 CSV 전처리

In [16]:
# 2024년 데이터만 사용, 필요한 컬럼만 추출
pop_cols = [
    '행정구역',
    '2024년_거주자_총인구수',
    '2024년_거주자_65~69세',
    '2024년_거주자_70~74세',
    '2024년_거주자_75~79세',
    '2024년_거주자_80~84세',
    '2024년_거주자_85~89세',
    '2024년_거주자_90~94세',
    '2024년_거주자_95~99세',
    '2024년_거주자_100세 이상',
]
pop_sub = pop[pop_cols].copy()

# 금천구 전체 행(첫 번째 행) 제외, 동 단위만 남기기
pop_sub = pop_sub[pop_sub['행정구역'].str.contains('동')]

# 숫자 컬럼 쉼표 제거 후 정수 변환
for col in pop_cols[1:]:
    pop_sub[col] = pop_sub[col].astype(str).str.replace(',', '').str.strip()
    pop_sub[col] = pd.to_numeric(pop_sub[col], errors='coerce')

# 65세 이상 합산
age_cols = [c for c in pop_cols if '세' in c or '이상' in c]
pop_sub['고령인구'] = pop_sub[age_cols].sum(axis=1)
pop_sub['총인구'] = pop_sub['2024년_거주자_총인구수']

# 행정동명 정리 (괄호 앞 이름만 추출)
pop_sub['행정동명'] = pop_sub['행정구역'].str.extract(r'금천구\s+(.+?)\s*\(')

# 결과 확인
print(pop_sub[['행정동명', '총인구', '고령인구']].to_string())

     행정동명    총인구  고령인구
1     가산동  24665  3038
2   독산제1동  46980  6892
3   독산제2동  17346  4057
4   독산제3동  22793  5314
5   독산제4동  14047  3311
6   시흥제1동  31476  7051
7   시흥제2동  20164  4763
8   시흥제3동  10101  2678
9   시흥제4동  18673  4846
10  시흥제5동  17564  4846


행정동 경계와 인구 데이터 결합 확인

In [17]:
# 행정동명 일치 여부 확인
print("경계 파일 동명:", dong['ADM_NM'].tolist())
print("\nCSV 동명:", pop_sub['행정동명'].tolist())

경계 파일 동명: ['가산동', '독산1동', '독산2동', '독산3동', '독산4동', '시흥1동', '시흥2동', '시흥3동', '시흥4동', '시흥5동']

CSV 동명: ['가산동', '독산제1동', '독산제2동', '독산제3동', '독산제4동', '시흥제1동', '시흥제2동', '시흥제3동', '시흥제4동', '시흥제5동']


동명 통일 후 재결합 -> 지표 1 계산

In [20]:
# CSV 동명에서 '제' 제거해서 경계 파일과 맞추기
pop_sub['행정동명_정리'] = pop_sub['행정동명'].str.replace('제', '', regex=False)

# 결합
dong_pop = dong.merge(
    pop_sub[['행정동명_정리', '총인구', '고령인구']],
    left_on='ADM_NM',
    right_on='행정동명_정리',
    how='left'
)

# 지표 1 계산
dong_pop['고령인구밀집도'] = dong_pop['고령인구'] / dong_pop['총인구']

print(dong_pop[['ADM_NM', '총인구', '고령인구', '고령인구밀집도']].to_string())

  ADM_NM    총인구  고령인구   고령인구밀집도
0    가산동  24665  3038  0.123170
1   독산1동  46980  6892  0.146701
2   독산2동  17346  4057  0.233887
3   독산3동  22793  5314  0.233142
4   독산4동  14047  3311  0.235709
5   시흥1동  31476  7051  0.224012
6   시흥2동  20164  4763  0.236213
7   시흥3동  10101  2678  0.265122
8   시흥4동  18673  4846  0.259519
9   시흥5동  17564  4846  0.275905




---



## <지표 2>
서울+안양 두 DEM 합치기


*   **안양 데이터를 포함한 이유**

금천구는 서울 남서쪽 끝에 위치하는데, '국토정보플랫폼'에서 다운로드한 서울 DEM 데이터 (37608) 단독으로는 금천구 남부 일부가 범위에서 벗어나게 되어 결국 전처리에서 오류가 발생함.

> 서울 도엽 y범위: 1944000~1972440

> 금천구 격자 y범위: 1937300~1943300

따라서 안양 도엽(37612)을 추가 병합하여 금천구 전체 영역을 커버하였음.

In [28]:
from rasterio.merge import merge

DEM_SEOUL_IMG = f'{BASE}/지표2_서울dem.img'
DEM_ANYANG_IMG = f'{BASE}/지표2_안양dem.img'
DEM_MERGED = f'{BASE}/dem_merged.tif'

with rasterio.open(DEM_SEOUL_IMG) as src1, \
     rasterio.open(DEM_ANYANG_IMG) as src2:
    mosaic, out_transform = merge([src1, src2])
    out_profile = src1.profile.copy()
    out_profile.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform
    })

with rasterio.open(DEM_MERGED, 'w', **out_profile) as dst:
    dst.write(mosaic)

print("DEM 합치기 완료")

with rasterio.open(DEM_MERGED) as src:
    print("합친 DEM 범위:", src.bounds)
    print("격자 범위:    ", grid_old.total_bounds)

DEM 합치기 완료
합친 DEM 범위: BoundingBox(left=933390.0, bottom=1916280.0, right=956340.0, top=1972440.0)
격자 범위:     [ 944400. 1937300.  949500. 1943300.]


☝️
> 합친 DEM y범위: 1916280 ~ 1972440

> 금천구 격자 y범위: 1937300 ~ 1943300  ← 완전히 안에 포함됨

✅ 따라서, 안양 데이터까지 포함하여 모든 범위 커버됨을 확인하였음.

경사도 계산 및 저장

In [29]:
SLOPE_TIF = f'{BASE}/slope.tif'

with rasterio.open(DEM_MERGED) as src:
    dem = src.read(1).astype(float)
    transform = src.transform
    res_x = transform[0]
    res_y = -transform[4]
    profile = src.profile.copy()
    nodata = src.nodata

if nodata is not None:
    dem[dem == nodata] = np.nan

dy, dx = np.gradient(dem, res_y, res_x)
slope_rad = np.arctan(np.sqrt(dx**2 + dy**2))
slope_deg = np.degrees(slope_rad)

print(f"경사도 최솟값: {np.nanmin(slope_deg):.2f}°")
print(f"경사도 최댓값: {np.nanmax(slope_deg):.2f}°")
print(f"경사도 평균값: {np.nanmean(slope_deg):.2f}°")

slope_profile = profile.copy()
slope_profile.update(driver="GTiff", dtype=rasterio.float32, count=1, nodata=-9999)

with rasterio.open(SLOPE_TIF, 'w', **slope_profile) as dst:
    slope_out = slope_deg.astype(np.float32)
    slope_out[np.isnan(slope_out)] = -9999
    dst.write(slope_out, 1)

print("경사도 래스터 저장 완료")

경사도 최솟값: 0.00°
경사도 최댓값: 52.84°
경사도 평균값: 5.93°
경사도 래스터 저장 완료


격자별 평균 경사도 계산

In [30]:
from rasterstats import zonal_stats

stats = zonal_stats(
    grid_old,
    SLOPE_TIF,
    stats=["mean"],
    nodata=-9999
)

grid_old['평균경사도'] = [s['mean'] if s['mean'] is not None else 0.0 for s in stats]
grid_tot['평균경사도'] = grid_old['평균경사도'].copy()

print("경사도 집계 완료")
print(grid_old[['gid', 'val', '평균경사도']].dropna(subset=['평균경사도']).head(10))

경사도 집계 완료
        gid  val     평균경사도
0  떎궗444429  NaN  0.537626
1  떎궗444430  NaN  1.294732
2  떎궗444431  NaN  1.110435
3  떎궗445427  NaN  1.136122
4  떎궗445428  NaN  1.276638
5  떎궗445429  NaN  1.614825
6  떎궗445430  NaN  2.120201
7  떎궗445431  NaN  1.182986
8  떎궗446424  NaN  1.877933
9  떎궗446425  NaN  1.459736


✅여기서 gid는 격자의 이름표(식별자) 역할인데, 어떤 계산에도 안 들어가고 그냥 행 식별용이라서 깨져도 결과에 영향이 없음.

✅val 열의 값들이 NaN으로 나오는 건 인구가 없는 격자(빈 땅, 공원 등)라서 정상임. 경사도 계산과는 별개.

혹시나 모두 NaN이 나오는 오류가 생긴 것일 수 있으니 아래에서 실제 인구가 있는 격자를 조회해봄.

In [31]:
# 실제로 인구 있는 격자만 확인

print(grid_old[grid_old['val'].notna()][['gid', 'val', '평균경사도']].head(10))

         gid   val     평균경사도
14  떎궗446430   0.0  1.439136
21  떎궗447427   0.0  0.084320
23  떎궗447429   0.0  1.877349
33  떎궗448427   6.0  0.090388
42  떎궗449422  70.0  0.881969
43  떎궗449423  11.0  0.274094
56  떎궗450420   0.0  1.110855
58  떎궗450422   0.0  0.175047
61  떎궗450425   0.0  0.413858
65  떎궗450429   0.0  0.183368


보행부담계수 α 부여

In [32]:
def get_alpha(slope):
    if slope is None or np.isnan(slope):
        return 1.0
    elif slope < 3:
        return 1.0
    elif slope < 8:
        return 1.3
    elif slope < 15:
        return 1.7
    else:
        return 2.0

grid_old['alpha'] = grid_old['평균경사도'].apply(get_alpha)

print("α 분포:")
print(grid_old['alpha'].value_counts().sort_index())

α 분포:
alpha
1.0    810
1.3    274
1.7    199
2.0    141
Name: count, dtype: int64


<보행부담계수 결과 해석>

α = 1.0 → 810개 격자 (평지, 경사 3° 미만)

α = 1.3 → 274개 격자 (완경사, 3~8°)

α = 1.7 → 199개 격자 (급경사, 8~15°)

α = 2.0 → 141개 격자 (보행 불가 수준, 15° 초과)

> 전체 1424개 격자 중 614개(43%)가 경사 있는 격자다. 금천구 남쪽에 삼성산, 호암산이 있어서 이 정도 비율은 현실적으로 맞다고 할 수 있다.

지표 2: 동별 보행 취약성 계산

In [35]:
# 격자에 행정동 정보 공간 조인
grid_old_proj = grid_old.copy()
dong_proj = dong_pop.copy()

# 격자 중심점 기준으로 행정동 귀속
grid_old_proj['geometry'] = grid_old_proj.geometry.centroid
grid_joined = gpd.sjoin(grid_old_proj, dong_proj[['ADM_NM', 'geometry']],
                         how='left', predicate='within')

# 인구 있는 격자만 사용
grid_valid = grid_joined[grid_joined['val'].notna()].copy()

# 동별 보행 취약성 계산
walkability = grid_valid.groupby('ADM_NM').apply(
    lambda x: x[x['alpha'] > 1.0]['val'].sum() / x['val'].sum()
    if x['val'].sum() > 0 else 0,
    include_groups=False
).reset_index()
walkability.columns = ['ADM_NM', '보행취약성']

print(walkability.to_string())


  ADM_NM     보행취약성
0    가산동  0.053267
1   독산1동  0.063083
2   독산2동  0.607119
3   독산3동  0.316914
4   독산4동  0.519231
5   시흥1동  0.410742
6   시흥2동  0.980326
7   시흥3동  0.739505
8   시흥4동  0.762599
9   시흥5동  0.464692


<결과 해석>

시흥2동  0.980  ← 고령인구 98%가 경사 있는 격자에 거주

시흥4동  0.763

시흥3동  0.740

독산2동  0.607

가산동   0.053  ← 거의 평지 (산업단지)

독산1동  0.063  ← 거의 평지
